In [7]:
import os

dataset_path = os.path.expanduser(
    "/Users/vjashwanth/Downloads/release_in_the_wild"
)

print(dataset_path)
print(os.listdir(dataset_path))

/Users/vjashwanth/Downloads/release_in_the_wild
['.DS_Store', 'test', 'attribution.txt', 'train', 'val']


In [17]:
import os
import glob

for split in ["train", "val", "test"]:
    print(f"\n📁 {split.upper()}")

    for label in ["fake", "real"]:
        files = glob.glob(
            os.path.join(dataset_path, split, label, "*.wav")
        )

        print(f"{label}: {len(files)} files")


📁 TRAIN
fake: 8271 files
real: 13974 files

📁 VAL
fake: 2363 files
real: 3992 files

📁 TEST
fake: 1182 files
real: 1997 files


In [18]:
import os
import glob
import random

random.seed(42)

# Get all training files
train_fake = glob.glob(
    os.path.join(dataset_path, "train", "fake", "*.wav")
)

train_real = glob.glob(
    os.path.join(dataset_path, "train", "real", "*.wav")
)

# Get validation files
val_fake = glob.glob(
    os.path.join(dataset_path, "val", "fake", "*.wav")
)

val_real = glob.glob(
    os.path.join(dataset_path, "val", "real", "*.wav")
)

# Create balanced samples
train_fake_sample = random.sample(train_fake, 2000)
train_real_sample = random.sample(train_real, 2000)

val_fake_sample = random.sample(val_fake, 500)
val_real_sample = random.sample(val_real, 500)

# Create datasets
train_data = (
    [(file, 0) for file in train_fake_sample] +
    [(file, 1) for file in train_real_sample]
)

val_data = (
    [(file, 0) for file in val_fake_sample] +
    [(file, 1) for file in val_real_sample]
)

# Shuffle
random.shuffle(train_data)
random.shuffle(val_data)

print("Training samples:", len(train_data))
print("Validation samples:", len(val_data))

Training samples: 4000
Validation samples: 1000


In [19]:
import librosa
import random

sample_path, label = random.choice(train_data)

audio, sr = librosa.load(
    sample_path,
    sr=None,
    mono=True
)

duration = len(audio) / sr

print("Audio file:", sample_path)
print("Label:", "FAKE" if label == 0 else "REAL")
print("Sample rate:", sr)
print("Duration:", round(duration, 2), "seconds")

Audio file: /Users/vjashwanth/Downloads/release_in_the_wild/train/real/8612.wav
Label: REAL
Sample rate: 16000
Duration: 8.58 seconds


In [20]:
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

model_name = "facebook/wav2vec2-base"

feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)

model = AutoModelForAudioClassification.from_pretrained(
    model_name,
    num_labels=2,
    label2id={
        "fake": 0,
        "real": 1
    },
    id2label={
        0: "fake",
        1: "real"
    },
    ignore_mismatched_sizes=True
)

print("✅ Wav2Vec2 model loaded!")
print(model.config.id2label)

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 17058.56it/s]
[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_q.bias               | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Wav2Vec2 model loaded!
{0: 'fake', 1: 'real'}


In [21]:
import torch
from torch.utils.data import Dataset
import librosa
import numpy as np

MAX_DURATION = 4  # seconds
TARGET_SR = 16000
MAX_LENGTH = TARGET_SR * MAX_DURATION


class AudioDataset(Dataset):
    
    def __init__(self, data, feature_extractor):
        self.data = data
        self.feature_extractor = feature_extractor
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        
        audio_path, label = self.data[idx]
        
        # Load audio
        audio, sr = librosa.load(
            audio_path,
            sr=TARGET_SR,
            mono=True
        )
        
        # Cut audio if longer than 4 seconds
        if len(audio) > MAX_LENGTH:
            audio = audio[:MAX_LENGTH]
        
        # Pad audio if shorter than 4 seconds
        if len(audio) < MAX_LENGTH:
            audio = np.pad(
                audio,
                (0, MAX_LENGTH - len(audio))
            )
        
        # Extract features
        inputs = feature_extractor(
            audio,
            sampling_rate=TARGET_SR,
            return_tensors="pt"
        )
        
        return {
            "input_values": inputs.input_values.squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [23]:
train_dataset = AudioDataset(
    train_data,
    feature_extractor
)

val_dataset = AudioDataset(
    val_data,
    feature_extractor
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Training dataset: 4000
Validation dataset: 1000


In [24]:
sample = train_dataset[0]

print("Input shape:", sample["input_values"].shape)
print("Label:", sample["labels"])

Input shape: torch.Size([64000])
Label: tensor(0)


In [25]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🍎 Apple Silicon GPU (MPS) is available!")
else:
    device = torch.device("cpu")
    print("⚠️ MPS not available. Using CPU.")

print("Device:", device)

🍎 Apple Silicon GPU (MPS) is available!
Device: mps


In [28]:
model = model.to(device)

print("Model moved to:", device)

Model moved to: mps


In [29]:
# Freeze the Wav2Vec2 feature extractor initially
for param in model.wav2vec2.parameters():
    param.requires_grad = False

print("✅ Wav2Vec2 base frozen")
print("✅ Training only classification layers")

✅ Wav2Vec2 base frozen
✅ Training only classification layers


In [30]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    small_train_dataset,
    batch_size=2,
    shuffle=True
)

val_loader = DataLoader(
    small_val_dataset,
    batch_size=2,
    shuffle=False
)

print("DataLoaders ready!")

DataLoaders ready!


In [32]:
import random

random.seed(42)

medium_train_data = (
    [(f, 0) for f in random.sample(train_fake, 500)] +
    [(f, 1) for f in random.sample(train_real, 500)]
)

medium_val_data = (
    [(f, 0) for f in random.sample(val_fake, 200)] +
    [(f, 1) for f in random.sample(val_real, 200)]
)

random.shuffle(medium_train_data)
random.shuffle(medium_val_data)

medium_train_dataset = AudioDataset(
    medium_train_data,
    feature_extractor
)

medium_val_dataset = AudioDataset(
    medium_val_data,
    feature_extractor
)

print("Training samples:", len(medium_train_dataset))
print("Validation samples:", len(medium_val_dataset))

Training samples: 1000
Validation samples: 400


In [33]:
from transformers import AutoModelForAudioClassification

model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=2,
    label2id={"fake": 0, "real": 1},
    id2label={0: "fake", 1: "real"},
    ignore_mismatched_sizes=True
)

model = model.to(device)

print("✅ Fresh model loaded!")

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 28130.01it/s]
[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_q.bias               | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Fresh model loaded!


In [34]:
for param in model.wav2vec2.parameters():
    param.requires_grad = False

print("✅ Base frozen")

✅ Base frozen


In [35]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    medium_train_dataset,
    batch_size=2,
    shuffle=True
)

val_loader = DataLoader(
    medium_val_dataset,
    batch_size=2,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 500
Validation batches: 200


In [36]:
import torch
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

print("✅ Optimizer ready")

✅ Optimizer ready


In [38]:
import torch
from tqdm.auto import tqdm
import copy

# ==========================================
# SETTINGS
# ==========================================

EPOCHS = 10

# Stop if validation accuracy does not improve
# for this many epochs
PATIENCE = 2

# Track best validation accuracy
best_val_accuracy = 0.0

# Counter for early stopping
patience_counter = 0

# Store best model weights
best_model_weights = None

# Training history
train_history = []
val_history = []
loss_history = []


# ==========================================
# TRAINING LOOP
# ==========================================

for epoch in range(EPOCHS):

    print("\n" + "=" * 60)
    print(f"🚀 STARTING EPOCH {epoch+1}/{EPOCHS}")
    print("=" * 60)

    # ==========================================
    # TRAINING MODE
    # ==========================================

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for batch in progress:

        # Move data to device
        input_values = batch["input_values"].to(device)
        labels = batch["labels"].to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_values=input_values
        )

        # Calculate loss
        loss = criterion(
            outputs.logits,
            labels
        )

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Track loss
        total_loss += loss.item()

        # Predictions
        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        # Calculate accuracy
        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        # Update progress bar
        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{correct/total:.4f}"
        )


    # ==========================================
    # TRAINING RESULTS
    # ==========================================

    train_loss = total_loss / len(train_loader)

    train_accuracy = correct / total


    # ==========================================
    # VALIDATION MODE
    # ==========================================

    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch in val_loader:

            # Move data to device
            input_values = batch["input_values"].to(device)
            labels = batch["labels"].to(device)

            # Forward pass
            outputs = model(
                input_values=input_values
            )

            # Predictions
            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            # Calculate validation accuracy
            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)


    # ==========================================
    # VALIDATION RESULTS
    # ==========================================

    val_accuracy = val_correct / val_total


    # ==========================================
    # SAVE HISTORY
    # ==========================================

    train_history.append(train_accuracy)

    val_history.append(val_accuracy)

    loss_history.append(train_loss)


    # ==========================================
    # PRINT RESULTS
    # ==========================================

    print("\n" + "=" * 60)

    print(f"📊 EPOCH {epoch+1}/{EPOCHS}")

    print(f"Training Loss:       {train_loss:.4f}")

    print(f"Training Accuracy:   {train_accuracy:.4f}")

    print(f"Validation Accuracy: {val_accuracy:.4f}")

    print("=" * 60)


    # ==========================================
    # EARLY STOPPING
    # ==========================================

    if val_accuracy > best_val_accuracy:

        print("🎉 Validation accuracy improved!")

        best_val_accuracy = val_accuracy

        patience_counter = 0

        # Save best model weights in memory
        best_model_weights = copy.deepcopy(
            model.state_dict()
        )

        # Also save to disk
        torch.save(
            model.state_dict(),
            "best_deepfake_audio_model.pth"
        )

        print(
            f"💾 Best model saved! "
            f"Validation Accuracy: {best_val_accuracy:.4f}"
        )

    else:

        patience_counter += 1

        print(
            f"⚠️ No improvement. "
            f"Patience: {patience_counter}/{PATIENCE}"
        )

        # Stop training if patience exceeded
        if patience_counter >= PATIENCE:

            print("\n🛑 EARLY STOPPING ACTIVATED!")
            print(
                f"Best Validation Accuracy: "
                f"{best_val_accuracy:.4f}"
            )

            break


# ==========================================
# RESTORE BEST MODEL
# ==========================================

if best_model_weights is not None:

    model.load_state_dict(
        best_model_weights
    )

    print("\n" + "=" * 60)

    print("✅ BEST MODEL RESTORED")

    print(
        f"🏆 Best Validation Accuracy: "
        f"{best_val_accuracy:.4f}"
    )

    print("=" * 60)


🚀 STARTING EPOCH 1/10


Epoch 1/10: 100%|██████████| 500/500 [10:59<00:00,  1.32s/it, accuracy=0.9050, loss=0.1239]



📊 EPOCH 1/10
Training Loss:       0.2437
Training Accuracy:   0.9050
Validation Accuracy: 0.8925
🎉 Validation accuracy improved!
💾 Best model saved! Validation Accuracy: 0.8925

🚀 STARTING EPOCH 2/10


Epoch 2/10: 100%|██████████| 500/500 [11:51<00:00,  1.42s/it, accuracy=0.9230, loss=0.1195]



📊 EPOCH 2/10
Training Loss:       0.1934
Training Accuracy:   0.9230
Validation Accuracy: 0.8675
⚠️ No improvement. Patience: 1/2

🚀 STARTING EPOCH 3/10


Epoch 3/10: 100%|██████████| 500/500 [11:49<00:00,  1.42s/it, accuracy=0.9210, loss=0.0681]



📊 EPOCH 3/10
Training Loss:       0.1943
Training Accuracy:   0.9210
Validation Accuracy: 0.8225
⚠️ No improvement. Patience: 2/2

🛑 EARLY STOPPING ACTIVATED!
Best Validation Accuracy: 0.8925

✅ BEST MODEL RESTORED
🏆 Best Validation Accuracy: 0.8925


In [39]:
import os
import glob
import random
from torch.utils.data import DataLoader

# ==========================================
# GET TEST FILES
# ==========================================

test_fake_files = glob.glob(
    os.path.join(dataset_path, "test", "fake", "*.wav")
)

test_real_files = glob.glob(
    os.path.join(dataset_path, "test", "real", "*.wav")
)

print("Fake test files:", len(test_fake_files))
print("Real test files:", len(test_real_files))

Fake test files: 1182
Real test files: 1997


In [40]:
random.seed(42)

NUM_TEST_SAMPLES = 500

selected_fake = random.sample(
    test_fake_files,
    NUM_TEST_SAMPLES
)

selected_real = random.sample(
    test_real_files,
    NUM_TEST_SAMPLES
)

test_files = []

for file in selected_fake:
    test_files.append((file, 0))  # fake

for file in selected_real:
    test_files.append((file, 1))  # real

random.shuffle(test_files)

print("Total test samples:", len(test_files))

Total test samples: 1000


In [42]:
test_dataset = AudioDataset(test_files)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

print("Test batches:", len(test_loader))

Test batches: 500


In [46]:
print("Test files:", len(test_files))
print("Test dataset:", len(test_dataset))
print("Test batches:", len(test_loader))

Test files: 1000
Test dataset: 1000
Test batches: 500


In [48]:
import torch
import torchaudio

In [52]:
import torch
import librosa
import numpy as np
from torch.utils.data import Dataset, DataLoader

class AudioDataset(Dataset):

    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        audio_path, label = self.files[idx]

        # Load audio using librosa — NOT torchaudio
        waveform, sample_rate = librosa.load(
            audio_path,
            sr=16000,
            mono=True
        )

        # Convert to PyTorch tensor
        waveform = torch.tensor(
            waveform,
            dtype=torch.float32
        )

        # Exactly 4 seconds
        max_length = 16000 * 4

        # Trim
        waveform = waveform[:max_length]

        # Pad
        if len(waveform) < max_length:
            waveform = torch.nn.functional.pad(
                waveform,
                (0, max_length - len(waveform))
            )

        return {
            "input_values": waveform,
            "labels": torch.tensor(label, dtype=torch.long)
        }

print("✅ New AudioDataset using LIBROSA created!")

✅ New AudioDataset using LIBROSA created!


In [53]:
test_dataset = AudioDataset(test_files)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

print("✅ Test dataset recreated!")
print("Test samples:", len(test_dataset))

✅ Test dataset recreated!
Test samples: 1000


In [54]:
sample = test_dataset[0]

print("Audio shape:", sample["input_values"].shape)
print("Label:", sample["labels"])

Audio shape: torch.Size([64000])
Label: tensor(1)


In [55]:
from tqdm.auto import tqdm

model.eval()

all_predictions = []
all_labels = []

correct = 0
total = 0

print("🧪 Starting test...")

with torch.no_grad():

    for batch in tqdm(test_loader, desc="Testing"):

        input_values = batch["input_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_values=input_values)

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total

print("\n" + "=" * 50)
print("🏆 FINAL TEST ACCURACY")
print("=" * 50)
print(f"Total samples: {total}")
print(f"Correct: {correct}")
print(f"Accuracy: {test_accuracy * 100:.2f}%")
print("=" * 50)

🧪 Starting test...


Testing: 100%|██████████| 500/500 [02:35<00:00,  3.22it/s]


🏆 FINAL TEST ACCURACY
Total samples: 1000
Correct: 865
Accuracy: 86.50%


In [58]:
import torch
import os

MODEL_PATH = "/Users/vjashwanth/Desktop/best_deepfake_audio_model.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "labels": {
        0: "fake",
        1: "real"
    },
    "sample_rate": 16000,
    "audio_duration": 4,
    "best_validation_accuracy": 0.8925
}, MODEL_PATH)

print("🎉 MODEL SAVED SUCCESSFULLY!")
print("📁 Location:", MODEL_PATH)

print("\nFile exists:", os.path.exists(MODEL_PATH))

🎉 MODEL SAVED SUCCESSFULLY!
📁 Location: /Users/vjashwanth/Desktop/best_deepfake_audio_model.pth

File exists: True
